[Guide Index](README.md)


# 01 Getting Started

StarLayer is a Python library for working with the emerging W3C 1.2 semantic specifications  (RDF 1.2, SPARQL 1.2 and SHACL 1.2.) Graphs are held by [Oxigraph](https://github.com/oxigraph/oxigraph), and validation is StarLayer's own SHACL engine.

StarLayer has no dependency on rdflib or pyshacl (rdflib is optional: `pip install "starlayer[rdflib]"` adds the conversion between rdflib graphs and StarLayer graphs, and the separate `starlayer-reasoning` package, for OWL reasoning, needs it too), and many of their functions and methods have been retained and extended to incorporate the 1.2 specifications.

This part of the guide installs StarLayer, builds a graph, queries it and validates it. Each section links to where the guide covers the topic in depth. It assumes a working knowledge of RDF, SPARQL and SHACL.

## How to run this notebook

Run the cells from top to bottom. Section 1 installs StarLayer if it is not installed, and the later sections reuse the graph built in section 2.

## 1. Install

Run the cell below. If StarLayer is not installed yet, the cell installs it from GitHub (it is not on PyPI, so `pip install starlayer` will not find it) and then stops with a message. When that happens, restart the kernel (Kernel > Restart) and run the notebook again from the top.

To update StarLayer, run `%pip uninstall -y starlayer` in a cell, restart the kernel, and run the install cell again. StarLayer follows the changes to the draft specifications, so a periodic update is advisable.

In [1]:
# Install StarLayer if it cannot be imported (the import also checks the pyoxigraph version).
# Inside a checkout of the repository it installs from the checkout, otherwise from GitHub.
try:
    import starlayer.graph  # noqa: F401
    print("starlayer is installed and current - skipping install.")
except ImportError as exc:
    import pathlib

    print(f"StarLayer needs installing or updating ({exc}).")
    checkout = pathlib.Path("../..").resolve()
    if (checkout / "packages" / "starlayer").is_dir():
        %pip install -e {checkout}/packages/starlayer
    else:
        %pip install --upgrade "git+https://github.com/hidden-graph/starlayer.git@shacl-engine#subdirectory=packages/starlayer"
    raise RuntimeError("StarLayer was installed: restart the kernel and run the notebook again.") from None

starlayer is installed and current - skipping install.


In [2]:
# Required imports for the examples below.
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import validate

# ex.alice is the IRI <http://example.org/alice>
ex = Namespace("http://example.org/")

## 2. Using `StarLayerGraph`

- `StarLayerGraph` holds a graph,
- `parse()` reads Turtle and the other RDF formats,
- `serialize()` writes a graph as text.

Parsing binds the prefixes the document declares. 

See [Graphs](02-graphs.ipynb) for discussion of RDF 1.2 in more depth.

In [3]:
# Build a graph from Turtle and write it back out.
g = StarLayerGraph()

g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

print("triples:", len(g))

print(g.serialize(format="turtle"))

triples: 3
@prefix ex: <http://example.org/> .
ex:alice ex:knows ex:bob ;
	ex:name "Alice" ;
	a ex:Person .



## 3. Querying

`query()` runs SPARQL and returns a `Result` that iterates over rows. 

A row gives each value by its variable name.  

See [SPARQL](03-sparql.ipynb) and [SPARQL 1.2 in queries](03a-sparql-12-queries.ipynb) for discussion of SPARQL 1.2 in more depth.

In [21]:
rows = g.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?friend WHERE { ?person ex:knows ?friend }
""")
for row in rows:
    friend = row.friend  # a value is read by its variable name
    print("the term:", friend)
    print(".value:  ", friend.value)

the term: <http://example.org/bob>
.value:   http://example.org/bob


## 4. Validating with SHACL

A shapes graph describes what data must look like. 

`validate(data_graph, shacl_graph=...)` checks a data graph against a shape graph and returns a result.  

See [SHACL shapes](04-shacl-shapes.ipynb) for discussion of SHACL 1.2 in more depth.

In [22]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

result = validate(g, shacl_graph=shapes)
print("conforms:", result.conforms)

# Take away Alice's name: the shape needs at least one.
g.remove((ex.alice, ex.name, None))
result = validate(g, shacl_graph=shapes)
print("conforms:", result.conforms)
print(result.report_text)

conforms: False
conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:minCount Literal("1", datatype=<http://www.w3.org/2001/XMLSchema#integer>) ; sh:path ex:name ]
	Focus Node: ex:alice
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name

